# PubChem Fetch -- Example

![PubChem Fetch -- Example](https://proto-bio.github.io/proto-assets/images/tool/pubchem/hero.png)

This notebook demonstrates `run_pubchem_fetch`, which resolves one of CID, name, SMILES, or InChIKey to canonical small-molecule structure data via the [PubChem PUG REST API](https://pubchem.ncbi.nlm.nih.gov/). Given any single identifier, the tool resolves it to a PubChem CID, fetches a configurable property bundle (formula, weight, SMILES, InChI, descriptors), and optionally returns a list of synonyms. See [Kim et al. 2023](https://doi.org/10.1093/nar/gkac956) for the underlying database.

In [1]:
from proto_tools.tools.database_retrieval import (
    PubChemFetchConfig,
    PubChemFetchInput,
    run_pubchem_fetch,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

Exactly one of the four identifier fields below must be provided.

In [2]:
# Display input docs
display_api_reference("pubchem", "input")

**Input** — `PubChemFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>cid</code> | <code>int &#124; None</code> | <code>None</code> | PubChem Compound Identifier |
| <code>name</code> | <code>str &#124; None</code> | <code>None</code> | Common or systematic name |
| <code>smiles</code> | <code>str &#124; None</code> | <code>None</code> | SMILES string |
| <code>inchi</code> | <code>str &#124; None</code> | <code>None</code> | Standard InChI string |
| <code>inchikey</code> | <code>str &#124; None</code> | <code>None</code> | Standard InChIKey |

## Config API Reference

In [3]:
# Display config docs
display_api_reference("pubchem", "config")

**Config** — `PubChemFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>properties</code> | <code>list[Literal['Title', 'MolecularFormula', 'MolecularFormulaNoCharge', 'MolecularWeight', 'SMILES', 'ConnectivitySMILES', 'InChI', 'InChIKey', 'IUPACName', 'ExactMass', 'MonoisotopicMass', 'XLogP', 'TPSA', 'Complexity', 'Charge', 'HBondDonorCount', 'HBondAcceptorCount', 'RotatableBondCount', 'HeavyAtomCount', 'AtomStereoCount', 'BondStereoCount', 'DefinedAtomStereoCount', 'DefinedBondStereoCount', 'UndefinedAtomStereoCount', 'UndefinedBondStereoCount', 'IsotopeAtomCount', 'CovalentUnitCount', 'PatentCount', 'PatentFamilyCount', 'AnnotationTypes', 'AnnotationTypeCount', 'SourceCategories', 'LiteratureCount', 'Volume3D', 'XStericQuadrupole3D', 'YStericQuadrupole3D', 'ZStericQuadrupole3D', 'FeatureCount3D', 'FeatureAcceptorCount3D', 'FeatureDonorCount3D', 'FeatureAnionCount3D', 'FeatureCationCount3D', 'FeatureRingCount3D', 'FeatureHydrophobeCount3D', 'ConformerModelRMSD3D', 'EffectiveRotorCount3D', 'ConformerCount3D', 'Fingerprint2D']]</code> | <code>['Title', 'MolecularFormula', 'MolecularWeight', 'SMILES', 'ConnectivitySMILES', 'InChI', 'InChIKey', 'IUPACName', 'ExactMass', 'TPSA', 'Complexity', 'Charge', 'HBondDonorCount', 'HBondAcceptorCount', 'RotatableBondCount', 'HeavyAtomCount']</code> | PubChem property names (Title, SMILES, MolecularWeight, XLogP, TPSA, ...) |
| <code>include_synonyms</code> | <code>bool</code> | <code>False</code> | Fetch synonyms (one extra HTTP call); truncated to 50 client-side |
| <code>include_description</code> | <code>bool</code> | <code>False</code> | Fetch textual descriptions of the compound (one extra HTTP call) |
| <code>include_aids</code> | <code>bool</code> | <code>False</code> | Fetch BioAssay IDs that tested this compound; can be thousands for common ones |

The default `properties` bundle covers structure, mass, and basic descriptor counts: `MolecularFormula`, `MolecularWeight`, `SMILES`, `ConnectivitySMILES`, `InChI`, `InChIKey`, `IUPACName`, `ExactMass`, `TPSA`, `Complexity`, `Charge`, `HBondDonorCount`, `HBondAcceptorCount`, `RotatableBondCount`, `HeavyAtomCount`.

## Output API Reference

In [4]:
# Display output docs
display_api_reference("pubchem", "output")

**Output** — `PubChemFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>cid</code> | <code>int</code> | required | Resolved PubChem CID |
| <code>all_matched_cids</code> | <code>list[int]</code> | <code>[]</code> | All CIDs returned by the resolver |
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Common compound name (e.g. 'Aspirin') |
| <code>molecular_formula</code> | <code>str &#124; None</code> | <code>None</code> | Molecular formula in Hill order |
| <code>molecular_weight</code> | <code>float &#124; None</code> | <code>None</code> | Molecular weight (g/mol) |
| <code>smiles</code> | <code>str &#124; None</code> | <code>None</code> | Canonical SMILES |
| <code>connectivity_smiles</code> | <code>str &#124; None</code> | <code>None</code> | Connectivity-only SMILES |
| <code>inchi</code> | <code>str &#124; None</code> | <code>None</code> | Standard InChI |
| <code>inchikey</code> | <code>str &#124; None</code> | <code>None</code> | Standard InChIKey |
| <code>iupac_name</code> | <code>str &#124; None</code> | <code>None</code> | IUPAC systematic name |
| <code>exact_mass</code> | <code>float &#124; None</code> | <code>None</code> | Exact (monoisotopic) mass in Da |
| <code>tpsa</code> | <code>float &#124; None</code> | <code>None</code> | Topological polar surface area in Å² |
| <code>complexity</code> | <code>int &#124; None</code> | <code>None</code> | PubChem molecular complexity score; higher values are more complex |
| <code>charge</code> | <code>int &#124; None</code> | <code>None</code> | Net formal charge (elementary charge units) |
| <code>hbond_donor_count</code> | <code>int &#124; None</code> | <code>None</code> | Hydrogen-bond donor count |
| <code>hbond_acceptor_count</code> | <code>int &#124; None</code> | <code>None</code> | Hydrogen-bond acceptor count |
| <code>rotatable_bond_count</code> | <code>int &#124; None</code> | <code>None</code> | Rotatable bond count |
| <code>heavy_atom_count</code> | <code>int &#124; None</code> | <code>None</code> | Non-hydrogen atom count |
| <code>synonyms</code> | <code>list[str]</code> | <code>[]</code> | Compound synonyms |
| <code>descriptions</code> | <code>list[str]</code> | <code>[]</code> | Textual descriptions of the compound |
| <code>bioassay_ids</code> | <code>list[int]</code> | <code>[]</code> | BioAssay IDs that tested this compound |
| <code>source_url</code> | <code>str</code> | required | URL of the property request |
| <code>raw_property_record</code> | <code>dict[str, Any]</code> | <code>{}</code> | Complete property record from PubChem for advanced access |

## Lookup by name

The most common starting point: pass a common name and let PubChem resolve it. Here we fetch aspirin and inspect the canonical structure plus a few descriptors.

In [5]:
inputs = PubChemFetchInput(name="aspirin")
config = PubChemFetchConfig()
output = run_pubchem_fetch(inputs, config)

print(f"CID: {output.cid}")
print(f"Molecular formula: {output.molecular_formula}")
print(f"Molecular weight: {output.molecular_weight}")
print(f"SMILES: {output.smiles}")
print(f"InChIKey: {output.inchikey}")
print(f"IUPAC name: {output.iupac_name}")
print(f"TPSA: {output.tpsa}")
print(f"H-bond donors: {output.hbond_donor_count}")
print(f"H-bond acceptors: {output.hbond_acceptor_count}")

CID: 2244
Molecular formula: C9H8O4
Molecular weight: 180.16
SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N
IUPAC name: 2-acetyloxybenzoic acid
TPSA: 63.6
H-bond donors: 1
H-bond acceptors: 4


## Lookup by CID

When you already know the PubChem CID, pass it directly to skip the resolver step. Here we fetch caffeine (CID 2519).

In [6]:
caffeine_output = run_pubchem_fetch(PubChemFetchInput(cid=2519), PubChemFetchConfig())

print(f"CID: {caffeine_output.cid}")
print(f"Molecular formula: {caffeine_output.molecular_formula}")
print(f"Molecular weight: {caffeine_output.molecular_weight}")
print(f"SMILES: {caffeine_output.smiles}")
print(f"IUPAC name: {caffeine_output.iupac_name}")

CID: 2519
Molecular formula: C8H10N4O2
Molecular weight: 194.19
SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C
IUPAC name: 1,3,7-trimethylpurine-2,6-dione


## Lookup by SMILES

PubChem can resolve a SMILES string back to its registered compound record. Here we round-trip aspirin's SMILES and verify the resolved CID.

In [7]:
smiles_output = run_pubchem_fetch(
    PubChemFetchInput(smiles="CC(=O)Oc1ccccc1C(=O)O"),
    PubChemFetchConfig(),
)

print(f"Resolved CID: {smiles_output.cid}")
print(f"Molecular formula: {smiles_output.molecular_formula}")
assert smiles_output.cid == 2244, f"Expected CID 2244 for aspirin, got {smiles_output.cid}"

Resolved CID: 2244
Molecular formula: C9H8O4


## Lookup by InChIKey

InChIKey is a fixed-length hash of the InChI string and is the most reliable identifier for exact-structure matching. Here we look up aspirin again via its InChIKey.

In [8]:
inchikey_output = run_pubchem_fetch(
    PubChemFetchInput(inchikey="BSYNRYMUTXBXSQ-UHFFFAOYSA-N"),
    PubChemFetchConfig(),
)

print(f"Resolved CID: {inchikey_output.cid}")
print(f"IUPAC name: {inchikey_output.iupac_name}")
assert inchikey_output.cid == 2244, f"Expected CID 2244 for aspirin, got {inchikey_output.cid}"

Resolved CID: 2244
IUPAC name: 2-acetyloxybenzoic acid


## Fetch with synonyms

Setting `include_synonyms=True` issues an extra HTTP request for the compound's synonym list. PubChem can return hundreds of synonyms per compound; the wrapper caps the returned list at 50. This is handy when you need to map between trade names, brand names, and chemical identifiers.

In [9]:
synonyms_output = run_pubchem_fetch(
    PubChemFetchInput(name="aspirin"),
    PubChemFetchConfig(include_synonyms=True),
)

print(f"CID: {synonyms_output.cid}")
print(f"Synonyms ({len(synonyms_output.synonyms)} of up to 50):")
for synonym in synonyms_output.synonyms[:10]:
    print(f"  - {synonym}")

CID: 2244
Synonyms (50 of up to 50):
  - aspirin
  - ACETYLSALICYLIC ACID
  - 50-78-2
  - 2-Acetoxybenzoic acid
  - 2-(Acetyloxy)benzoic acid
  - Acylpyrin
  - O-Acetylsalicylic acid
  - o-Acetoxybenzoic acid
  - Acenterine
  - Acetophen


## Custom property subset

When you only need a few fields, restrict the `properties` list to keep the response small. Here we ask only for formula, weight, and SMILES.

In [10]:
subset_output = run_pubchem_fetch(
    PubChemFetchInput(name="aspirin"),
    PubChemFetchConfig(properties=["MolecularFormula", "MolecularWeight", "SMILES"]),
)

print(f"Molecular formula: {subset_output.molecular_formula}")
print(f"Molecular weight: {subset_output.molecular_weight}")
print(f"SMILES: {subset_output.smiles}")

Molecular formula: C9H8O4
Molecular weight: 180.16
SMILES: CC(=O)OC1=CC=CC=C1C(=O)O


## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [11]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("aspirin_pubchem", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'aspirin_pubchem.json'}")

Exported to example_output/aspirin_pubchem.json
